# SnowMap Analytics: POC de linguagem natural para SQL e gráficos

Adaptação segura do princípio do Chat2VIS: a LLM conhece o esquema, traduz a pergunta para SQL e escolhe uma visualização. O código de renderização é local e confiável; nenhum código arbitrário retornado pela LLM é executado.

## 1. Configuração
Use `ollama` localmente ou `openai` no Colab. Para OpenAI, defina `OPENAI_API_KEY` no ambiente antes de executar.

In [ ]:
import json, os, re, sqlite3, tempfile, urllib.request, urllib.error
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

LLM_PROVIDER = os.getenv("SNOWMAP_LLM_PROVIDER", "ollama")  # ollama | openai
default_model = "llama3.2:3b" if LLM_PROVIDER == "ollama" else "gpt-4o-mini"
LLM_MODEL = os.getenv("SNOWMAP_LLM_MODEL", default_model)
OLLAMA_URL = os.getenv("OLLAMA_URL", "http://localhost:11434/api/chat")
OPENAI_URL = os.getenv("OPENAI_URL", "https://api.openai.com/v1/chat/completions")
MAX_ROWS = 500
print(f"Provedor: {LLM_PROVIDER} | Modelo: {LLM_MODEL}")

## 2. Banco acadêmico mockado
O modelo normalizado permite perguntas sobre artigos, autores, anos, venues, citações, acesso aberto e decisão de triagem.

In [ ]:
db_path = Path(tempfile.gettempdir()) / "snowmap_analytics_poc.sqlite"
conn = sqlite3.connect(db_path)
conn.executescript("""
DROP TABLE IF EXISTS paper_authors; DROP TABLE IF EXISTS authors; DROP TABLE IF EXISTS papers;
CREATE TABLE papers (
  paper_id INTEGER PRIMARY KEY, title TEXT NOT NULL, year INTEGER NOT NULL,
  venue TEXT NOT NULL, citation_count INTEGER NOT NULL, open_access INTEGER NOT NULL,
  screening_status TEXT NOT NULL CHECK(screening_status IN ('included','excluded','unsure'))
);
CREATE TABLE authors (author_id INTEGER PRIMARY KEY, name TEXT NOT NULL, affiliation TEXT);
CREATE TABLE paper_authors (paper_id INTEGER, author_id INTEGER, author_order INTEGER,
  PRIMARY KEY (paper_id, author_id), FOREIGN KEY(paper_id) REFERENCES papers(paper_id),
  FOREIGN KEY(author_id) REFERENCES authors(author_id));
""")
papers = [
 (1,'Reliable Forward Snowballing',2021,'EASE',42,1,'included'),
 (2,'Automating Study Selection',2021,'IST',31,0,'included'),
 (3,'LLMs for Evidence Synthesis',2022,'EMSE',67,1,'included'),
 (4,'Citation Network Exploration',2022,'JSS',24,1,'excluded'),
 (5,'Human in the Loop Reviews',2022,'EASE',18,0,'unsure'),
 (6,'Open Science Mapping',2023,'IST',53,1,'included'),
 (7,'Visual Analytics for SLRs',2023,'EMSE',39,1,'included'),
 (8,'Reproducible Snowballing',2023,'JSS',28,0,'included'),
 (9,'Prompting for Paper Screening',2024,'EASE',21,1,'unsure'),
 (10,'Knowledge Graphs in Reviews',2024,'IST',35,1,'included'),
 (11,'Evaluating LLM Review Assistants',2024,'EMSE',16,0,'excluded'),
 (12,'Adaptive Literature Discovery',2025,'JSS',9,1,'included')
]
authors = [(1,'Ana Silva','PUC-Rio'),(2,'Bruno Lima','USP'),(3,'Carla Souza','UFMG'),(4,'Diego Costa','UFPE'),(5,'Elena Martins','PUC-Rio'),(6,'Felipe Rocha','USP'),(7,'Giulia Alves','UFMG'),(8,'Hugo Nunes','UFPE')]
links = [(1,1,1),(1,2,2),(2,2,1),(2,3,2),(2,4,3),(3,1,1),(3,3,2),(3,5,3),(3,7,4),(4,4,1),(4,6,2),(5,5,1),(5,8,2),(6,1,1),(6,6,2),(6,7,3),(7,2,1),(7,5,2),(7,8,3),(8,3,1),(8,4,2),(9,1,1),(9,2,2),(9,5,3),(9,8,4),(10,3,1),(10,6,2),(10,7,3),(11,4,1),(11,5,2),(12,1,1),(12,6,2),(12,8,3)]
conn.executemany('INSERT INTO papers VALUES (?,?,?,?,?,?,?)', papers)
conn.executemany('INSERT INTO authors VALUES (?,?,?)', authors)
conn.executemany('INSERT INTO paper_authors VALUES (?,?,?)', links)
conn.commit()
pd.read_sql_query('SELECT * FROM papers LIMIT 5', conn)

## 3. Esquema, prompts e chamada à LLM

In [ ]:
def database_schema(connection):
    descriptions = {
      'papers.open_access': '1 significa acesso aberto; 0 significa fechado',
      'papers.screening_status': 'included, excluded ou unsure',
      'paper_authors.author_order': 'posição do autor na lista do artigo'
    }
    parts = []
    tables = connection.execute("SELECT name FROM sqlite_master WHERE type='table' ORDER BY name").fetchall()
    for (table,) in tables:
        columns = connection.execute(f'PRAGMA table_info({table})').fetchall()
        fields = [f'{c[1]} {c[2]}' for c in columns]
        parts.append(f"{table}({', '.join(fields)})")
    return '\n'.join(parts) + '\n' + '\n'.join(f'{k}: {v}' for k,v in descriptions.items())

SCHEMA = database_schema(conn)
print(SCHEMA)

def call_llm(system_prompt, user_prompt):
    messages = [{'role':'system','content':system_prompt},{'role':'user','content':user_prompt}]
    if LLM_PROVIDER == 'ollama':
        payload = {'model': LLM_MODEL, 'messages': messages, 'stream': False, 'format': 'json', 'options': {'temperature': 0}}
        headers = {'Content-Type':'application/json'}
        url = OLLAMA_URL
    elif LLM_PROVIDER == 'openai':
        key = os.environ.get('OPENAI_API_KEY')
        if not key: raise RuntimeError('Defina OPENAI_API_KEY antes de usar o provedor openai.')
        payload = {'model': LLM_MODEL, 'messages': messages, 'temperature': 0}
        headers = {'Content-Type':'application/json','Authorization':f'Bearer {key}'}
        url = OPENAI_URL
    else: raise ValueError('LLM_PROVIDER deve ser ollama ou openai')
    request = urllib.request.Request(url, data=json.dumps(payload).encode(), headers=headers, method='POST')
    try:
        with urllib.request.urlopen(request, timeout=120) as response: data = json.load(response)
    except urllib.error.URLError as exc:
        raise RuntimeError(f'Não foi possível chamar a LLM em {url}: {exc}') from exc
    return data['message']['content'] if LLM_PROVIDER == 'ollama' else data['choices'][0]['message']['content']

SQL_SYSTEM = '''Você traduz perguntas sobre revisão sistemática para SQLite.
Retorne APENAS JSON válido: {"sql":"...","explanation":"..."}.
Gere exatamente uma consulta SELECT (CTE WITH é permitida), nunca altere dados.
Use aliases legíveis em português e CAST(... AS REAL) quando uma média exigir divisão.
Para quantidade de autores por artigo, use paper_authors e COUNT(author_id).'''

CHART_SYSTEM = '''Você é especialista em visualização de dados científicos.
Retorne APENAS JSON válido com: chart_type, x, y, title, x_label, y_label.
chart_type deve ser bar, line, scatter, pie ou histogram.
x e y devem ser nomes exatos das colunas fornecidas; y pode ser null apenas para histogram.
Prefira barras para categorias, linha para tempo, scatter para relação numérica e histograma para distribuição.'''

def extract_json(text):
    text = re.sub(r'^```(?:json)?\s*|\s*```$', '', text.strip(), flags=re.I|re.S)
    try: return json.loads(text)
    except json.JSONDecodeError:
        match = re.search(r'\{.*\}', text, re.S)
        if not match: raise ValueError(f'A LLM não retornou JSON: {text}')
        return json.loads(match.group())

## 4. Validação do SQL e renderização segura

In [ ]:
FORBIDDEN = re.compile(r'\b(insert|update|delete|drop|alter|create|replace|attach|detach|pragma|vacuum|reindex|analyze)\b', re.I)

def validate_sql(sql):
    cleaned = sql.strip().rstrip(';').strip()
    if ';' in cleaned: raise ValueError('Apenas uma instrução SQL é permitida.')
    if not re.match(r'^(select|with)\b', cleaned, re.I): raise ValueError('A consulta deve começar com SELECT ou WITH.')
    if FORBIDDEN.search(cleaned): raise ValueError('A consulta contém uma operação não permitida.')
    return cleaned

def run_read_only_query(connection, sql):
    safe_sql = validate_sql(sql)
    allowed = {sqlite3.SQLITE_SELECT, sqlite3.SQLITE_READ, sqlite3.SQLITE_FUNCTION}
    if hasattr(sqlite3, 'SQLITE_RECURSIVE'): allowed.add(sqlite3.SQLITE_RECURSIVE)
    def authorizer(action, arg1, arg2, db_name, trigger):
        return sqlite3.SQLITE_OK if action in allowed else sqlite3.SQLITE_DENY
    connection.set_authorizer(authorizer)
    try: return pd.read_sql_query(f'SELECT * FROM ({safe_sql}) AS result LIMIT {MAX_ROWS}', connection)
    finally: connection.set_authorizer(None)

def render_chart(df, spec):
    required = {'chart_type','x','y','title','x_label','y_label'}
    if not required.issubset(spec): raise ValueError(f'Especificação incompleta: {required - set(spec)}')
    chart_type, x, y = spec['chart_type'], spec['x'], spec['y']
    if chart_type not in {'bar','line','scatter','pie','histogram'}: raise ValueError('Tipo de gráfico não permitido.')
    if x not in df.columns or (y is not None and y not in df.columns): raise ValueError(f'Colunas inválidas. Disponíveis: {list(df.columns)}')
    fig, ax = plt.subplots(figsize=(10,5))
    if chart_type == 'bar': ax.bar(df[x].astype(str), df[y], color='#176D6D')
    elif chart_type == 'line': ax.plot(df[x], df[y], marker='o', color='#176D6D')
    elif chart_type == 'scatter': ax.scatter(df[x], df[y], color='#176D6D', s=60)
    elif chart_type == 'pie': ax.pie(df[y], labels=df[x].astype(str), autopct='%1.1f%%')
    elif chart_type == 'histogram': ax.hist(df[x].dropna(), bins='auto', color='#176D6D', edgecolor='white')
    ax.set_title(spec['title']); ax.set_xlabel(spec['x_label']); ax.set_ylabel(spec['y_label'])
    if chart_type in {'bar','line'}: ax.tick_params(axis='x', rotation=30)
    if chart_type != 'pie':
        ax.spines['top'].set_visible(False); ax.spines['right'].set_visible(False); ax.grid(axis='y', alpha=.2)
    fig.tight_layout(); return fig

def analyze(question, show_prompts=False):
    sql_prompt = f'ESQUEMA DO BANCO:\n{SCHEMA}\n\nPERGUNTA DO USUÁRIO:\n{question}'
    sql_answer = extract_json(call_llm(SQL_SYSTEM, sql_prompt))
    sql = validate_sql(sql_answer['sql'])
    result = run_read_only_query(conn, sql)
    if result.empty: raise ValueError('A consulta não retornou dados.')
    metadata = {c: str(result[c].dtype) for c in result.columns}
    chart_prompt = f'Pergunta: {question}\nColunas e tipos: {json.dumps(metadata, ensure_ascii=False)}\nAmostra: {result.head(5).to_json(orient="records", force_ascii=False)}'
    chart_spec = extract_json(call_llm(CHART_SYSTEM, chart_prompt))
    fig = render_chart(result, chart_spec)
    if show_prompts: print('SQL prompt:', sql_prompt, '\nChart prompt:', chart_prompt)
    print('SQL:', sql); print('Explicação:', sql_answer.get('explanation','')); print('Gráfico:', chart_spec)
    display(result)
    return {'question':question,'sql':sql,'data':result,'chart_spec':chart_spec,'figure':fig}


## 5. Experimento ponta a ponta
Troque a pergunta por qualquer análise compatível com o esquema. O resultado contém o SQL auditável, a tabela, a especificação e a figura.

In [ ]:
question = 'Mostre a média de autores por artigo em cada ano.'
result = analyze(question)
result['figure'].savefig('snowmap_analytics_result.png', dpi=300, bbox_inches='tight')
plt.show()

## Perguntas adicionais para avaliação

- Quantos artigos incluídos existem por venue?
- Mostre a evolução do número de artigos por ano.
- Qual é a média de citações dos artigos abertos e fechados?
- Mostre os cinco autores com maior soma de citações.
- Mostre a distribuição da quantidade de autores por artigo.

Antes de integrar ao SnowMap, substitua o SQLite pelo banco real, forneça ao prompt apenas tabelas permitidas e execute consultas com um usuário de banco estritamente read-only.